# **Lab 4: Implementing BST**

Part 1: Describe what gets printed by the following code

In [1]:
class BstNode:
    def __init__(self, key):
        self.key = key
        self.right = None
        self.left = None

    def display(self):
        lines, *_ = self._display_aux()
        for line in lines:
            print(line)

    def _display_aux(self):
        """Returns list of strings, width, height, and horizontal coordinate of the root."""

        # No child.
        if self.right is None and self.left is None:
            line = '%s' % self.key
            width = len(line)
            height = 1
            middle = width // 2
            return [line], width, height, middle

        # Only left child.
        if self.right is None:
            lines, n, p, x = self.left._display_aux()
            s = '%s' % self.key
            u = len(s)
            first_line = (x + 1) * ' ' + (n - x - 1) * '_' + s
            second_line = x * ' ' + '/' + (n - x - 1 + u) * ' '
            shifted_lines = [line + u * ' ' for line in lines]
            return [first_line, second_line] + shifted_lines, n + u, p + 2, n + u // 2

        # Only right child.
        if self.left is None:
            lines, n, p, x = self.right._display_aux()
            s = '%s' % self.key
            u = len(s)
            first_line = s + x * '_' + (n - x) * ' '
            second_line = (u + x) * ' ' + '\\' + (n - x - 1) * ' '
            shifted_lines = [u * ' ' + line for line in lines]
            return [first_line, second_line] + shifted_lines, n + u, p + 2, u // 2

        # Two children.
        left, n, p, x = self.left._display_aux()
        right, m, q, y = self.right._display_aux()
        s = '%s' % self.key
        u = len(s)

        first_line = (x + 1) * ' ' + (n - x - 1) * '_' + s + \
                     y * '_' + (m - y) * ' '

        second_line = x * ' ' + '/' + \
                      (n - x - 1 + u + y) * ' ' + '\\' + \
                      (m - y - 1) * ' '

        if p < q:
            left += [n * ' '] * (q - p)
        elif q < p:
            right += [m * ' '] * (p - q)

        zipped_lines = zip(left, right)
        lines = [first_line, second_line] + \
                [a + u * ' ' + b for a, b in zipped_lines]

        return lines, n + m + u, max(p, q) + 2, n + u // 2

In [2]:
a = BstNode(3)

a.left = BstNode(5)

a.left.right = BstNode(1)

a.display()

print("------------")

b = BstNode(7)

b.left = BstNode(4)

a.right = b

a.display()

print("------------")

b = BstNode(2)

b.right = a.right

a.right = b

a.display()

 _3
/  
5  
 \ 
 1 
------------
 _3_ 
/   \
5   7
 \ / 
 1 4 
------------
 _3   
/  \  
5  2_ 
 \   \
 1   7
    / 
    4 


Explanation: The code manually creates the nodes and connects them using "left" & "right". The second connects the objects together without actually using BST insetion. Which means that the resulting trees dont actually obey the Binary Search Tree ordering rule.

Part 2: BST Implementation

In [3]:
class BST(BstNode):

    def __init__(self, key):
        super().__init__(key)

    # ----------------------------------------------------
    # FIND
    # ----------------------------------------------------
    def find(self, value):

        # Value found
        if value == self.key:
            return True

        # Search the left subtree
        elif value < self.key:
            if self.left is None:
                return False

            return self.left.find(value)

        # Search the right subtree
        else:
            if self.right is None:
                return False

            return self.right.find(value)

    # ----------------------------------------------------
    # INSERT
    # ----------------------------------------------------
    def insert(self, value):

        # Smaller values go left
        if value < self.key:

            if self.left is None:
                self.left = BST(value)

            else:
                self.left.insert(value)

        # Larger values go right
        elif value > self.key:

            if self.right is None:
                self.right = BST(value)

            else:
                self.right.insert(value)

        # Duplicate values are ignored
        return self

    # ----------------------------------------------------
    # FIND MINIMUM
    # Helper used during removal
    # ----------------------------------------------------
    def get_minimum(self):

        current = self

        while current.left is not None:
            current = current.left

        return current

    # ----------------------------------------------------
    # REMOVE
    # ----------------------------------------------------
    def remove(self, value):

        # Search left
        if value < self.key:

            if self.left is not None:
                self.left = self.left.remove(value)

        # Search right
        elif value > self.key:

            if self.right is not None:
                self.right = self.right.remove(value)

        # Node found
        else:

            # CASE 1 / CASE 2:
            # No left child.
            # Return the right child, which may also be None.
            if self.left is None:
                return self.right

            # CASE 2:
            # No right child.
            # Replace this node with its left child.
            if self.right is None:
                return self.left

            # CASE 3:
            # Node has two children.
            # Find the inorder successor:
            # smallest value in the right subtree.
            successor = self.right.get_minimum()

            # Copy the successor's value into this node.
            self.key = successor.key

            # Remove the original successor node.
            self.right = self.right.remove(successor.key)

        return self

In [4]:
tree = BST(8)

values = [3, 10, 1, 6, 14, 4, 7, 13]

for value in values:
    tree.insert(value)

print("BST after inserting values:")
tree.display()

print("\nFinding values:")
print("Find 7:", tree.find(7))
print("Find 13:", tree.find(13))
print("Find 99:", tree.find(99))

print("\nRemoving 3:")
tree = tree.remove(3)
tree.display()

print("\nRemoving root 8:")
tree = tree.remove(8)
tree.display()

BST after inserting values:
  ___8_     
 /     \    
 3_   10___ 
/  \       \
1  6      14
  / \    /  
  4 7   13  

Finding values:
Find 7: True
Find 13: True
Find 99: False

Removing 3:
  __8_     
 /    \    
 4   10___ 
/ \       \
1 6      14
   \    /  
   7   13  

Removing root 8:
  __10___ 
 /       \
 4      14
/ \    /  
1 6   13  
   \      
   7      


The implementation extends "BestNode" and adds three operations which are "find(value)", "insert(value)", and "remove(value)". These are in charge searching for values, placing a new value, and deleting a value while preserving the BST property.

When a node with two children is removed, its inorder successor is used. The inorder successor is the smallest value in the node's right subtree.